In [ ]:
import csv
import os

if not os.path.exists("test_notebooks"):
    os.chdir("..")

assert os.path.exists("test_notebooks")

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from modelscope import snapshot_download


In [ ]:
from pathlib import Path


tags_txt = "entropy/conf/tag_datasets/danbooru.txt"
tags = Path(tags_txt).read_text("utf8").split("\n")
tags = [p.strip() for p in tags]
tags = [p for p in tags if p]

tasg = list(dict.fromkeys(tags))

tags = [{"id": str(i), "tag": p} for i, p in enumerate(tags)]

print("tags", len(tags))
print("sample", tags[:8])

# to id

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
model_dir = snapshot_download("BAAI/bge-m3", local_dir="./ai_models/BAAI/bge-m3", ignore_file_pattern="onnx/*")

In [ ]:
embedding_model = SentenceTransformer(
    model_dir, 
    device=device, local_files_only=True)

In [ ]:
# build embedding database
client = chromadb.PersistentClient(path="./database/chroma_1")

In [ ]:
# delete collection
try:
    client.delete_collection(name="danbooru_tags")
except:
    pass

In [ ]:
# create
collection = client.get_or_create_collection(name="danbooru_tags",metadata={"hnsw:space": "cosine"})

In [ ]:
from tqdm import tqdm
from more_itertools import chunked


batch_size = 128 

batches = list(chunked(tags, batch_size))

for batch in tqdm(batches):
    # print("batch", len(batch))
    batch_tags = [p['tag'] for p in batch]
    batch_ids = [str(p['id']) for p in batch]
    
    # normalize_embeddings=True 对余弦相似度检索非常重要
    with torch.no_grad():
        embeddings = embedding_model.encode(
            batch_tags, 
            batch_size=batch_size, 
            normalize_embeddings=True
        ).tolist()

    
    # 插入 ChromaDB
    collection.add(
        embeddings=embeddings,
        documents=batch_tags,
        ids=batch_ids
    )

print(f"成功导入 {collection.count()} 个标签")

In [ ]:
# test

query_text = "火药"
query_embedding = embedding_model.encode(query_text, normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

print("检索到的标签:", results['documents'])